# Signal Lab Quickstart

Notebook-first exploration of one concrete copy-trade signal: `sig_val_opp_flipper`.

This notebook keeps the important logic explicit so you can inspect:
- raw trades and train/val/test splits
- wallet metrics and copy-universe selection
- candidate BUY trades
- flipper cohort definition
- direct signal construction
- before/after trade quality after thresholding

Main stage1-style objects exposed here: `df_full`, `df_train`, `df_val`, `df_test`, `candidate_trades`, `c_train`, `c_val`, `c_test`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd().resolve()
ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'signal_lab' else NOTEBOOK_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lib import (
    DEFAULT_TAGS,
    compute_copyable_notional,
    compute_opening_metrics,
    load_trades,
    split_data,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics
from signal_lab.signal_engines import PositionSignalEngine, archetype_sets, compute_hold_time_metrics
from signal_lab.signal_lib import (
    apply_rank_transformer,
    bootstrap_ic,
    compute_event_ic,
    evaluate_strategy,
    fit_rank_transformer,
    fit_roi_residualizer,
    residualized_roi,
)
from signal_lab.stage1 import evaluate_threshold_grid


## Parameters

In [2]:
COPY_MIN_BUY_ROI = 0.02
COPY_MIN_BUCKETS = 20
COPY_MIN_MARKETS = 15
COPY_MIN_TRADE_COUNT = 100
COPY_MAX_DD_TO_PNL = 0.6
COPY_MIN_COPYABLE_ROI = 0.05
ARCH_MIN_TRADE_COUNT = 100
COST_BPS = 0.0
THRESHOLD_MIN_TRADES = 20

ENGINE_COLS = [
    'wallet', 'condition_id', 'outcome', 'dt',
    'side', 'position', 'quantity', 'price',
]

SIGNAL_COL = 'sig_val_opp_flipper'
COPY_SIGNAL_COL = 'sig_copy_anti_crowding_flipper'
SCORE_COL = 'score_flipper'


## 1. Load raw trades and split them

In [3]:
df_full = load_trades(tags=DEFAULT_TAGS)
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

print(f'df_full={len(df_full):,} train={len(df_train):,} val={len(df_val):,} test={len(df_test):,}')


Markets: 1974837


Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...


Total trades loaded: 14,250,603


Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00


Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z


Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)


  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)


  Total: 14,250,603 trades  (56,875 markets)
df_full=14,250,603 train=4,337,961 val=5,211,194 test=4,701,448


## 2. Compute wallet metrics directly in the notebook

In [4]:
wallet_metrics, _ = compute_wallet_metrics(df_train)
wallet_metrics['copyable_pnl_factor'] = np.clip(
    wallet_metrics['copyable_pnl'] / wallet_metrics['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_metrics['copyable_roi'] = wallet_metrics['average_roi'] * wallet_metrics['copyable_pnl_factor']

opening_metrics = compute_opening_metrics(df_train)
wallet_metrics = wallet_metrics.merge(opening_metrics, on='wallet', how='left')
for col in ['opening_roi', 'opening_pnl', 'opening_copyable_roi', 'opening_copyable_pnl']:
    wallet_metrics[col] = wallet_metrics[col].fillna(0.0)

hold_metrics = compute_hold_time_metrics(df_train)

display(wallet_metrics[['wallet', 'buy_roi', 'copyable_roi', 'trade_count', 'num_markets', 'num_buckets']].head())


,wallet,buy_roi,copyable_roi,trade_count,num_markets,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.008757,-0.000000,53,16,38
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.039375,0.069611,210,180,208
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,-0.070000,0.000000,2,1,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.028754,0.243228,79,13,34
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,-0.009430,-0.219729,348,240,299


## 3. Select the copy universe directly in the notebook

In [5]:
copy_mask = (
    (wallet_metrics['buy_roi'] >= COPY_MIN_BUY_ROI)
    & (wallet_metrics['num_buckets'] >= COPY_MIN_BUCKETS)
    & (wallet_metrics['num_markets'] >= COPY_MIN_MARKETS)
    & (wallet_metrics['trade_count'] >= COPY_MIN_TRADE_COUNT)
    & (wallet_metrics['max_drawdown_to_pnl'].fillna(1.0) <= COPY_MAX_DD_TO_PNL)
    & (wallet_metrics['copyable_roi'].fillna(0.0) >= COPY_MIN_COPYABLE_ROI)
)

copy_wallets = set(wallet_metrics.loc[copy_mask, 'wallet'])
copy_wallet_table = wallet_metrics.loc[copy_mask].copy()

print(f'copy_wallets={len(copy_wallets)}')
display(copy_wallet_table[['wallet', 'buy_roi', 'copyable_roi', 'trade_count', 'num_markets', 'num_buckets']].head(10))


copy_wallets=58


,wallet,buy_roi,copyable_roi,trade_count,num_markets,num_buckets
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.039375,0.069611,210,180,208
39,0x07d601375c9bbb9037ad3c7a8f8fa0deff8164fb,0.083860,0.054305,491,89,353
52,0x0a59a7f2a870392a5f555aaa11f49d612e748e5c,0.037250,0.097951,1459,651,1106
59,0x0af29cc14fb231218c8d005685ac3db427a1ce86,0.088851,0.101813,1559,82,886
66,0x0d1048ec2a35bb690b217181bf5859d772b9a713,0.105692,0.055301,491,105,340
90,0x10484477985de3c6e9869f104f2d13b59a8ce362,0.141095,0.056424,517,155,476
123,0x177500541ae20bb0d46ab0db3fd2559e2a7e85b0,0.080500,0.059502,804,62,463
152,0x1c72101b1db0c7f4447edf999c6701bde226f5ce,0.042375,0.083309,305,81,194
158,0x1d4811eb053f27006701d9b852541c672e2bba3c,0.040698,0.192799,100,24,76
170,0x1e7220f4ed4e8f654a2743c961a43b675ae7bd5d,0.076560,0.126778,178,63,156


## 4. Build candidate copyable BUY trades and residualized ROI

In [6]:
candidate_trades = df_full[
    df_full['wallet'].isin(copy_wallets) & (df_full['side'] == 'BUY')
].copy()

c_train, c_val, c_test = split_data(candidate_trades, method='chronological')
fit = fit_roi_residualizer(c_train['copyable_roi'], c_train['price'])
for frame in (c_train, c_val, c_test):
    frame['roi_res'] = residualized_roi(frame['copyable_roi'], frame['price'], fit)

conditions = set(candidate_trades['condition_id'].unique())
print(f'candidate_trades={len(candidate_trades):,} conditions={len(conditions):,}')
display(c_train[['wallet', 'condition_id', 'outcome', 'price', 'copyable_pnl', 'copyable_roi', 'roi_res']].head())


Chronological split: train <= 2026-05-31T00:00:00Z, val <= 2026-06-27T00:00:00Z, test > 2026-06-27T00:00:00Z
Method: chronological  |  Unique end dates: 92  (train=36, val=27, test=29)

  Train:     46,338 trades  (7,717 markets)
  Val:       52,217 trades  (10,200 markets)
  Test:      42,247 trades  (8,990 markets)
  Total:    140,802 trades  (26,907 markets)


candidate_trades=140,802 conditions=26,907


,wallet,condition_id,outcome,price,copyable_pnl,copyable_roi,roi_res
25,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x02336b1d5f9d52c8998bd56a57f4969f233fd0d92b8c...,Yes,0.170,0.000000e+00,NaN,0.380996
26,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x037f7df5081789ebca056ad6e272452931c6a1375c2e...,No,0.929,1.261213e-16,0.076426,-0.198300
27,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x07e7c570f4d38fa8786947312850b4fc9af50860ddc0...,Yes,0.009,0.000000e+00,NaN,0.422127
28,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,No,0.790,1.680000e+00,0.265823,-0.059108
29,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,No,0.760,7.741932e-01,0.315789,-0.040447


## 5. Define the archetype sets and inspect the flipper cohort

In [7]:
signal_sets = {
    name: set(sel['wallet'])
    for name, sel in archetype_sets(wallet_metrics, hold_metrics, min_trade_count=ARCH_MIN_TRADE_COUNT).items()
}

flipper_wallets = signal_sets['flipper']
flipper_table = wallet_metrics[wallet_metrics['wallet'].isin(flipper_wallets)].merge(
    hold_metrics[['wallet', 'median_hold_min', 'median_flip_min', 'n_round_trips', 'round_trip_rate']],
    on='wallet',
    how='left',
)

print(f'flipper_wallets={len(flipper_wallets)}')
display(flipper_table[['wallet', 'trade_count', 'buy_roi', 'copyable_roi', 'median_flip_min', 'n_round_trips']].head(10))


flipper_wallets=161


,wallet,trade_count,buy_roi,copyable_roi,median_flip_min,n_round_trips
0,0x0377e02b4ed7aa13fd9442e80a6362198489ae31,507,0.046816,0.000000,3.733333,32
1,0x058327e8fec1fdf218c4c2c20f3a30db18887dd4,362,0.012862,-0.011056,0.191667,135
2,0x09a85cfe4d51e0cde12daa4d255b3f789fb447c1,554,0.030790,-0.013971,5.850000,31
3,0x0a59a7f2a870392a5f555aaa11f49d612e748e5c,1459,0.037250,0.097951,15.750000,53
4,0x0af29cc14fb231218c8d005685ac3db427a1ce86,1559,0.088851,0.101813,16.500000,203
5,0x0e0cc261b9315683bbac3ebdd0a5be75fc29dde1,166,0.032844,0.152234,14.400000,24
6,0x0f37cb80dee49d55b5f6d9e595d52591d6371410,131,-0.017913,-0.173196,0.050000,25
7,0x10968af4773b31a1d97e1a4f072376de12cfa162,663,0.022484,0.000000,2.766667,72
8,0x10ab807ac93681f1f8d567501721ce245ec5b2e8,179,0.017900,-0.000000,15.666667,44
9,0x116db6298abcdefe06f9f5458c293c7de185fbf1,3564,0.007165,-0.000000,4.208333,208


## 6. Build the flipper signal directly

`sig_val_opp_flipper` comes from these two steps:
1. `A_flipper, B_flipper = engine.build_set(flipper_wallets, conditions=conditions)`
2. `engine.attach_position_signals(...)` which writes `sig_val_opp_flipper` to each candidate-trade frame.

In [8]:
restricted = df_full[df_full['condition_id'].isin(conditions)][ENGINE_COLS].copy()
engine = PositionSignalEngine(restricted)
A_flipper, B_flipper = engine.build_set(flipper_wallets, conditions=conditions)

for frame in (c_train, c_val, c_test):
    engine.attach_position_signals(frame, 'flipper', A_flipper, B_flipper)

display(A_flipper.head())
display(B_flipper.head())
display(c_train[['wallet', 'condition_id', 'outcome', 'price', 'sig_val_opp_flipper']].head(10))


,dt,cum_pos,cum_vac,condition_id,outcome
2760496,2025-12-16 14:09:17+00:00,5.000000,0.750000,0xe6aaeaed1eaba5c98787cd168c98bfb519d013275d76...,Yes
1100882,2025-12-16 15:02:49+00:00,10.000000,1.400000,0x5ec1564e0902e14b11aef7a09e163d6ee9f6539da2f4...,Yes
2760497,2025-12-25 09:00:55+00:00,7.870194,1.180529,0xe6aaeaed1eaba5c98787cd168c98bfb519d013275d76...,Yes
2760498,2025-12-25 21:46:49+00:00,7.870388,1.180558,0xe6aaeaed1eaba5c98787cd168c98bfb519d013275d76...,Yes
1415562,2026-04-28 00:00:22+00:00,7.620000,1.905000,0x7b68a6316a598c7cc317a71f3dc5252a772bab7f9f43...,Yes


,dt,cum_pos,cum_vac,condition_id,outcome
2509442,2025-12-25 09:00:55+00:00,5.000000,0.750000,0xe6aaeaed1eaba5c98787cd168c98bfb519d013275d76...,Yes
2509443,2025-12-25 21:46:49+00:00,7.870194,1.180529,0xe6aaeaed1eaba5c98787cd168c98bfb519d013275d76...,Yes
1287631,2026-04-28 00:00:30+00:00,2.840908,0.340909,0x788e9b9ef17412d626f43df347b104252e7ad8be0c95...,Yes
952978,2026-04-28 00:00:58+00:00,2.969694,1.009696,0x599ea6499b7b07d4ed48a3fe85b5d07b64881de359d4...,Yes
1287632,2026-04-28 00:00:58+00:00,7.831816,0.939818,0x788e9b9ef17412d626f43df347b104252e7ad8be0c95...,Yes


,wallet,condition_id,outcome,price,sig_val_opp_flipper
25,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x02336b1d5f9d52c8998bd56a57f4969f233fd0d92b8c...,Yes,0.170,157.275857
26,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x037f7df5081789ebca056ad6e272452931c6a1375c2e...,No,0.929,2.567784
27,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x07e7c570f4d38fa8786947312850b4fc9af50860ddc0...,Yes,0.009,0.000000
28,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,No,0.790,1.305300
29,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0a4ed7975dab87ca68a9b4cd7fbe12589a657cc4b493...,No,0.760,2.137900
30,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x025532d8c6ec28e36be21cf97b40a7df532539023d64...,No,0.650,0.049992
31,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0e106e57f99efd8ab24de60c7b1266fb08bb39b8d306...,Yes,0.015,0.000000
32,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0593dd667f07a366c4d4e5a31abf261a2e16b82710c5...,No,0.978,0.000000
33,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x0ffc7b17c17eb4cc788555108528d3609abeb8a763f3...,No,0.672,2.640000
34,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x00388e88ae0beea218fc23948124df8bbd3a74ce89df...,No,0.980,0.000000


## 7. Turn it into a copy-trade signal and measure presence

In [9]:
for frame in (c_train, c_val, c_test):
    frame[COPY_SIGNAL_COL] = -frame[SIGNAL_COL].fillna(0.0)

presence_df = pd.DataFrame([
    {
        'split': label,
        'signal_gt_zero_share': float((frame[COPY_SIGNAL_COL] > 0).mean()),
        'signal_nonzero_share': float((frame[COPY_SIGNAL_COL] != 0).mean()),
        'mean_signal': float(frame[COPY_SIGNAL_COL].mean()),
        'IC_copyable_roi': compute_event_ic(frame[COPY_SIGNAL_COL], frame['copyable_roi']),
        'IC_roi_res': compute_event_ic(frame[COPY_SIGNAL_COL], frame['roi_res']),
    }
    for label, frame in [('train', c_train), ('val', c_val), ('test', c_test)]
])

pooled = pd.concat([c_train[[COPY_SIGNAL_COL, 'copyable_roi', 'roi_res']], c_val[[COPY_SIGNAL_COL, 'copyable_roi', 'roi_res']]], ignore_index=True)
boot_mean_roi, boot_lo_roi, boot_hi_roi = bootstrap_ic(pooled[COPY_SIGNAL_COL], pooled['copyable_roi'], n_iter=500, alpha=0.05, seed=42)
boot_mean_res, boot_lo_res, boot_hi_res = bootstrap_ic(pooled[COPY_SIGNAL_COL], pooled['roi_res'], n_iter=500, alpha=0.05, seed=42)

display(presence_df.round(4))
print(f'pooled train+val bootstrap IC on copyable_roi: mean={boot_mean_roi:+.4f} ci=({boot_lo_roi:+.4f}, {boot_hi_roi:+.4f})')
print(f'pooled train+val bootstrap IC on roi_res:      mean={boot_mean_res:+.4f} ci=({boot_lo_res:+.4f}, {boot_hi_res:+.4f})')


,split,signal_gt_zero_share,signal_nonzero_share,mean_signal,IC_copyable_roi,IC_roi_res
0,train,0.0,0.8959,-193.6091,0.0708,0.1831
1,val,0.0,0.8878,-127.4682,0.0413,0.1892
2,test,0.0,0.8999,-131.5172,0.0320,0.1378


pooled train+val bootstrap IC on copyable_roi: mean=+0.0577 ci=(+0.0503, +0.0655)
pooled train+val bootstrap IC on roi_res:      mean=+0.1801 ci=(+0.1741, +0.1859)


## 8. Fit a train-only score transform and choose a threshold on validation

In [10]:
score_fit = fit_rank_transformer(c_train[COPY_SIGNAL_COL].fillna(0.0))
for frame in (c_train, c_val, c_test):
    frame[SCORE_COL] = apply_rank_transformer(frame[COPY_SIGNAL_COL].fillna(0.0), score_fit)

val_grid = evaluate_threshold_grid(c_val, SCORE_COL, cost_bps=COST_BPS)
best_val = val_grid[val_grid['trades'] >= THRESHOLD_MIN_TRADES]
if best_val.empty:
    best_val = val_grid
best_row = best_val.sort_values('copyable_pnl_net', ascending=False).iloc[0]
best_threshold = float(best_row['threshold'])

display(val_grid.sort_values('copyable_pnl_net', ascending=False).head(10).round(4))
print(f'best_threshold={best_threshold:+.2f}')


,threshold,trades,copyable_pnl,copyable_roi,copyable_pnl_net,copyable_roi_net,total_pnl,notional,copyable_notional,firing_rate,cost_paid,pnl_per_trade_net
2,-0.90,50572,4204.7068,0.0202,4204.7068,0.0202,32269.3037,937901.9264,207816.1778,0.9685,0.0,0.0831
3,-0.85,49623,4110.8743,0.0203,4110.8743,0.0203,32780.2933,922185.7543,202239.8597,0.9503,0.0,0.0828
4,-0.80,48451,3025.8293,0.0155,3025.8293,0.0155,31466.4228,900796.0632,195708.6532,0.9279,0.0,0.0625
7,-0.65,45190,2892.2543,0.0165,2892.2543,0.0165,28727.4252,829309.2115,175270.6863,0.8654,0.0,0.0640
6,-0.70,46368,2741.5065,0.0149,2741.5065,0.0149,30539.8534,859037.0428,183524.4478,0.8880,0.0,0.0591
10,-0.50,41636,2729.9048,0.0173,2729.9048,0.0173,26174.5614,769198.9833,157826.0767,0.7974,0.0,0.0656
9,-0.55,42978,2547.6471,0.0155,2547.6471,0.0155,25916.0145,791106.2921,164779.6941,0.8231,0.0,0.0593
5,-0.75,47346,2533.6314,0.0133,2533.6314,0.0133,30649.4346,876948.0945,189812.8685,0.9067,0.0,0.0535
8,-0.60,44142,2461.4457,0.0144,2461.4457,0.0144,27071.7217,813183.6029,170693.5666,0.8454,0.0,0.0558
1,-0.95,51562,2365.5813,0.0110,2365.5813,0.0110,29254.0209,951822.5331,214673.5136,0.9875,0.0,0.0459


best_threshold=-0.90


## 9. Findings: all candidate BUYs vs signal-filtered BUYs

This is the main before/after view.

- `all_candidates`: copy every candidate BUY in the copy universe
- `selected_by_signal`: copy only trades whose score clears the validation-chosen threshold

In [11]:
def summarize_split(label, frame):
    baseline = evaluate_strategy(frame, SCORE_COL, -np.inf, cost_bps=COST_BPS)
    selected = evaluate_strategy(frame, SCORE_COL, best_threshold, cost_bps=COST_BPS)
    return [
        {
            'split': label,
            'group': 'all_candidates',
            'trades': baseline['trades'],
            'copyable_notional': baseline['copyable_notional'],
            'copyable_pnl_net': baseline['copyable_pnl_net'],
            'copyable_roi_net': baseline['copyable_roi_net'],
            'pnl_per_trade_net': baseline['copyable_pnl_net'] / max(baseline['trades'], 1),
            'firing_rate': baseline['firing_rate'],
        },
        {
            'split': label,
            'group': 'selected_by_signal',
            'trades': selected['trades'],
            'copyable_notional': selected['copyable_notional'],
            'copyable_pnl_net': selected['copyable_pnl_net'],
            'copyable_roi_net': selected['copyable_roi_net'],
            'pnl_per_trade_net': selected['copyable_pnl_net'] / max(selected['trades'], 1),
            'firing_rate': selected['firing_rate'],
        },
        {
            'split': label,
            'group': 'delta_selected_minus_all',
            'trades': selected['trades'] - baseline['trades'],
            'copyable_notional': selected['copyable_notional'] - baseline['copyable_notional'],
            'copyable_pnl_net': selected['copyable_pnl_net'] - baseline['copyable_pnl_net'],
            'copyable_roi_net': selected['copyable_roi_net'] - baseline['copyable_roi_net'],
            'pnl_per_trade_net': (selected['copyable_pnl_net'] / max(selected['trades'], 1)) - (baseline['copyable_pnl_net'] / max(baseline['trades'], 1)),
            'firing_rate': selected['firing_rate'] - baseline['firing_rate'],
        },
    ]

findings = pd.DataFrame(
    summarize_split('train', c_train)
    + summarize_split('val', c_val)
    + summarize_split('test', c_test)
)
display(findings.round(4))


,split,group,trades,copyable_notional,copyable_pnl_net,copyable_roi_net,pnl_per_trade_net,firing_rate
0,train,all_candidates,46338,274635.7344,29562.4908,0.1076,0.6380,1.0000
1,train,selected_by_signal,44021,253826.7385,27603.8081,0.1088,0.6271,0.9500
2,train,delta_selected_minus_all,-2317,-20808.9960,-1958.6828,0.0011,-0.0109,-0.0500
3,val,all_candidates,52217,217015.6102,2070.1273,0.0095,0.0396,1.0000
4,val,selected_by_signal,50572,207816.1778,4204.7068,0.0202,0.0831,0.9685
5,val,delta_selected_minus_all,-1645,-9199.4324,2134.5795,0.0107,0.0435,-0.0315
6,test,all_candidates,42247,205129.2990,9605.3161,0.0468,0.2274,1.0000
7,test,selected_by_signal,41030,195401.6291,8749.2404,0.0448,0.2132,0.9712
8,test,delta_selected_minus_all,-1217,-9727.6699,-856.0757,-0.0020,-0.0141,-0.0288


## 10. Trade-level examples

Inspect actual candidate trades with signal, score, and whether they would be copied after thresholding.

In [12]:
test_view = c_test[[
    'dt', 'wallet', 'condition_id', 'outcome', 'price',
    'copyable_notional', 'copyable_pnl', 'copyable_roi', 'roi_res',
    SIGNAL_COL, COPY_SIGNAL_COL, SCORE_COL,
]].copy()
test_view['copy_trade'] = test_view[SCORE_COL] >= best_threshold

display(test_view.sort_values(SCORE_COL, ascending=False).head(2).round(4))
display(test_view[test_view['copy_trade']].sort_values('copyable_pnl', ascending=False).head(2).round(4))


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_26970/3036782631.py:8: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(test_view.sort_values(SCORE_COL, ascending=False).head(2).round(4))


,dt,wallet,condition_id,outcome,price,copyable_notional,copyable_pnl,copyable_roi,roi_res,sig_val_opp_flipper,sig_copy_anti_crowding_flipper,score_flipper,copy_trade
2449,2026-06-26 12:23:47+00:00,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0x07b6225fbff9e3b260cb159d3d059f90f70dbbe5418d...,Yes,0.340,-0.0,0.0,NaN,0.4083,0.0,-0.0,0.8959,True
11583251,2026-06-29 12:23:26+00:00,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0xd830f30523e88a9e2068b8c79a1ceb380662994ed1d9...,No,0.994,0.0,0.0,NaN,1.4631,0.0,-0.0,0.8959,True


/var/folders/j8/0dbnwk8n6m933m843h7hb88w0000gn/T/ipykernel_26970/3036782631.py:9: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(test_view[test_view['copy_trade']].sort_values('copyable_pnl', ascending=False).head(2).round(4))


,dt,wallet,condition_id,outcome,price,copyable_notional,copyable_pnl,copyable_roi,roi_res,sig_val_opp_flipper,sig_copy_anti_crowding_flipper,score_flipper,copy_trade
1401157,2026-07-25 06:18:49+00:00,0xa89518aca5a633a79ad1e9737209c9689f83faac,0x1a15b9db7aa08ff03f4c15471c8f6c404293c69b9d46...,No,0.0895,60.0939,611.0145,10.1677,0.4109,17.3808,-17.3808,0.4746,True
14011683,2026-07-14 04:55:17+00:00,0xc60bb547dbdf59c8d1746bc8a754accc3809d9f7,0xf2c9a5f2c7adf9cd8d7605d7616f5c5dbc3a048d8f54...,Yes,0.2329,86.5505,285.0406,3.2933,0.3557,61.4960,-61.4960,0.0461,True


## 11. Summary verdict

In [13]:
test_baseline = findings[(findings['split'] == 'test') & (findings['group'] == 'all_candidates')].iloc[0]
test_selected = findings[(findings['split'] == 'test') & (findings['group'] == 'selected_by_signal')].iloc[0]
val_baseline = findings[(findings['split'] == 'val') & (findings['group'] == 'all_candidates')].iloc[0]
val_selected = findings[(findings['split'] == 'val') & (findings['group'] == 'selected_by_signal')].iloc[0]
test_ic_row = presence_df[presence_df['split'] == 'test'].iloc[0]
val_ic_row = presence_df[presence_df['split'] == 'val'].iloc[0]

summary = pd.DataFrame([
    {'metric': 'validation IC on copyable_roi', 'value': round(float(val_ic_row['IC_copyable_roi']), 4)},
    {'metric': 'test IC on copyable_roi', 'value': round(float(test_ic_row['IC_copyable_roi']), 4)},
    {'metric': 'validation IC on roi_res', 'value': round(float(val_ic_row['IC_roi_res']), 4)},
    {'metric': 'test IC on roi_res', 'value': round(float(test_ic_row['IC_roi_res']), 4)},
    {'metric': 'validation ROI improvement', 'value': round(float(val_selected['copyable_roi_net'] - val_baseline['copyable_roi_net']), 4)},
    {'metric': 'test ROI improvement', 'value': round(float(test_selected['copyable_roi_net'] - test_baseline['copyable_roi_net']), 4)},
    {'metric': 'validation firing rate', 'value': round(float(val_selected['firing_rate']), 4)},
    {'metric': 'test firing rate', 'value': round(float(test_selected['firing_rate']), 4)},
])
display(summary)

verdict = (
    f"Signal direction looks consistent by IC on copyable ROI "
    f"(val={val_ic_row['IC_copyable_roi']:+.4f}, test={test_ic_row['IC_copyable_roi']:+.4f}), "
    f"and also on roi_res (val={val_ic_row['IC_roi_res']:+.4f}, test={test_ic_row['IC_roi_res']:+.4f}). "
    f"But the current threshold rule does not improve test copyable ROI "
    f"(all={test_baseline['copyable_roi_net']:.4f}, selected={test_selected['copyable_roi_net']:.4f}, "
    f"delta={test_selected['copyable_roi_net'] - test_baseline['copyable_roi_net']:+.4f}). "
    f"This suggests the signal has ranking information, but the current near-copy-all threshold "
    f"(best_threshold={best_threshold:+.2f}, firing_rate={test_selected['firing_rate']:.1%}) is too weak or unstable out of sample."
)
print(verdict)


,metric,value
0,validation IC on copyable_roi,0.0413
1,test IC on copyable_roi,0.0320
2,validation IC on roi_res,0.1892
3,test IC on roi_res,0.1378
4,validation ROI improvement,0.0107
5,test ROI improvement,-0.0020
6,validation firing rate,0.9685
7,test firing rate,0.9712


Signal direction looks consistent by IC on copyable ROI (val=+0.0413, test=+0.0320), and also on roi_res (val=+0.1892, test=+0.1378). But the current threshold rule does not improve test copyable ROI (all=0.0468, selected=0.0448, delta=-0.0020). This suggests the signal has ranking information, but the current near-copy-all threshold (best_threshold=-0.90, firing_rate=97.1%) is too weak or unstable out of sample.
